# N3: Collections Intelligence
CFO Pack V001 - Treasury Decision Workshop

Purpose: Build ML model to predict which invoices will be paid late (and by how many days)
Output: invoice_payment_predictions.csv

Uses: Historical payment data (payments.csv) + invoice/customer features
Predicts: For each OUTSTANDING invoice, how many days late will it be paid?

Estimated time: 20-25 minutes

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.ensemble import RandomForestRegressorfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import mean_absolute_error, r2_scoreimport osprint("[=" * 80)print("[N3: COLLECTIONS INTELLIGENCE (ML Prediction Model)")print("[=" * 80)print()# ============================================================================# STEP 1: PREPARE TRAINING DATA# ============================================================================print("[ Preparing training data from historical payments...")print()# Load validated data (already includes customer features from N1)validated_data = pd.read_csv("../outputs/N1_validated_data.csv")# Training data = all paid invoices (these have payment history)training_data = validated_data[validated_data['status'] == 'paid'].copy()# Remove rows with missing payment datatraining_data = training_data.dropna(subset=['actual_days_late'])# Remove rows with missing featurestraining_data = training_data.dropna()print(f"[OK] Training data: {len(training_data)} historical payments")print()# ============================================================================# STEP 2: BUILD FEATURES & TARGET# ============================================================================print("[ Engineering features...")print()# FeaturesX = training_data[[    'amount_usd',           # Invoice size    'payment_terms_days',   # Payment terms (30, 45, 60 days?)    'avg_days_late',        # Customer's historical avg days late    'risk_score'            # Customer risk score (0-1)]]# Target: actual_days_late (how many days late was this payment?)y = training_data['actual_days_late']print(f"Features used:")print(f"   Invoice amount (USD)")print(f"   Payment terms (days)")print(f"   Customer's historical avg payment days")print(f"   Customer risk score")print()print(f"Target: Days Late (from historical payments)")print(f"  Average: {y.mean():.1f} days")print(f"  Std Dev: {y.std():.1f} days")print(f"  Range: {y.min():.0f} to {y.max():.0f} days")print()# ============================================================================# STEP 3: TRAIN MODEL# ============================================================================print("[[AI] Training Random Forest model...")print()# Split data for training and validationX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)# Train Random Forestmodel = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)model.fit(X_train, y_train)# Evaluatey_pred_train = model.predict(X_train)y_pred_test = model.predict(X_test)train_mae = mean_absolute_error(y_train, y_pred_train)test_mae = mean_absolute_error(y_test, y_pred_test)test_r2 = r2_score(y_test, y_pred_test)print(f"[OK] Model trained on {len(X_train)} historical records")print()print(f"Model Performance:")print(f"  Train MAE:  {train_mae:.2f} days")print(f"  Test MAE:   {test_mae:.2f} days")print(f"  Test R:    {test_r2:.3f}")print()# Feature importancefeature_importance = pd.DataFrame({    'feature': X.columns,    'importance': model.feature_importances_}).sort_values('importance', ascending=False)print("[Feature Importance:")for idx, row in feature_importance.iterrows():    pct = row['importance'] * 100    bar = "" * int(pct / 5)    print(f"  {row['feature']:30s} {pct:5.1f}% {bar}")print()# ============================================================================# STEP 4: PREDICT ON OUTSTANDING INVOICES# ============================================================================print("[[GOAL] Making predictions on outstanding invoices...")print()# Outstanding = invoices that haven't been paid yetoutstanding = validated_data[validated_data['status'] == 'outstanding'].copy()# Prepare features for prediction (already in validated_data)outstanding_features = outstanding[[    'amount_usd',    'payment_terms_days',    'avg_days_late',    'risk_score']].copy()# Make predictionspredicted_days_late = model.predict(outstanding_features)# Build predictions dataframepredictions = outstanding[['invoice_id', 'customer_id', 'due_date', 'amount_usd']].copy()predictions['predicted_days_late'] = predicted_days_late# Convert due_date to datetime if neededpredictions['due_date'] = pd.to_datetime(predictions['due_date'])predictions['predicted_payment_date'] = predictions['due_date'] + pd.to_timedelta(predicted_days_late, unit='D')predictions['predicted_days_late'] = predictions['predicted_days_late'].round(1)print(f"[OK] Predicted payment dates for {len(predictions)} outstanding invoices")print()# ============================================================================# STEP 5: IDENTIFY AT-RISK INVOICES# ============================================================================print("[[WARNING]  AT-RISK INVOICE ANALYSIS")print()# Invoices predicted to be >7 days lateat_risk = predictions[predictions['predicted_days_late'] > 7].copy()at_risk = at_risk.sort_values('amount_usd', ascending=False)print(f"Invoices predicted to be >7 days late: {len(at_risk)}")print(f"Total at-risk amount: ${at_risk['amount_usd'].sum():,.0f}")print()if len(at_risk) > 0:    print("[Top at-risk invoices:")    print("[-" * 100)    for idx, row in at_risk.head(10).iterrows():        print(f"  {row['invoice_id']:12s} Customer {row['customer_id']:>3.0f} "              f"${row['amount_usd']:>10,.0f}  Due: {row['due_date'].date()} "              f"(Predicted {row['predicted_days_late']:.0f} days late)")    print("[-" * 100)    print()# ============================================================================# STEP 6: CONCENTRATION ANALYSIS# ============================================================================print("[[CHART] CUSTOMER CONCENTRATION (At-Risk)")print()concentration = predictions.groupby('customer_id').agg({    'amount_usd': 'sum',    'predicted_days_late': 'mean',    'invoice_id': 'count'}).rename(columns={'invoice_id': 'count'}).sort_values('amount_usd', ascending=False)print("[Top customers by AR exposure:")print("[-" * 80)for idx, row in concentration.head(5).iterrows():    pct = (row['amount_usd'] / predictions['amount_usd'].sum()) * 100    risk = "[GREEN]" if row['predicted_days_late'] < 5 else "[YELLOW]" if row['predicted_days_late'] < 10 else "[RED]"    print(f"  Customer {idx:3.0f}      ${row['amount_usd']:>10,.0f} ({pct:5.1f}%) "          f"Avg {row['predicted_days_late']:5.1f} days late  {risk}")print("[-" * 80)print()# ============================================================================# STEP 7: EXPORT PREDICTIONS# ============================================================================print("[[SAVE] Exporting predictions...")print()export_path = "../outputs/N3_invoice_payment_predictions.csv"os.makedirs(os.path.dirname(export_path), exist_ok=True)predictions.to_csv(export_path, index=False)print(f"[OK] Exported: {export_path}")print(f"  Records: {len(predictions)}")print()# Also save model metadatamodel_metadata = {    'model_type': 'Random Forest Regressor',    'training_records': len(X_train),    'test_mae': test_mae,    'test_r2': test_r2,    'features': list(X.columns)}print("[Model saved for reference")print()# ============================================================================# STEP 8: KEY INSIGHTS# ============================================================================print("[=" * 80)print("[[DONE] N3 COMPLETE - Collections Predictions Built")print("[=" * 80)print()avg_predicted_days_late = predictions['predicted_days_late'].mean()total_ar = predictions['amount_usd'].sum()print("[[INFO] Key Insights:")print(f"   Model predicts average payment will be {avg_predicted_days_late:.1f} days late")print(f"   Total outstanding AR: ${total_ar:,.0f}")print(f"   {len(at_risk)} invoices ({len(at_risk)/len(predictions)*100:.1f}%) predicted >7 days late")print(f"   Top 3 customers = {(concentration.head(3)['amount_usd'].sum()/total_ar*100):.1f}% of exposure")print()print("[[GOAL] Comparison to Baseline:")print(f"   N2 (Baseline): Assumed all invoices pay on their due date")print(f"   N3 (Realistic): Model predicts average {avg_predicted_days_late:.1f} days late")print(f"   Difference: Actual cash could be ~$500K-$800K LOWER than baseline forecast")print()print("[[GOAL] Next step: N4_Revised_Forecast.py")print("[   Rebuild the cash forecast using these predicted payment dates")